# 03_generate_patients.ipynb

## Cell 1 — Markdown

```markdown
# 🏥 Hospital Analytics Portfolio Project
## Patient Dimension Data Generation

This notebook generates synthetic patient master data for the
Hospital Analytics Portfolio Project.

Output:
    dim_patient.csv

The generated patient dimension contains demographic, contact,
emergency-contact, registration, and status information.
```

In [15]:
# ==============================================================================
# IMPORT REQUIRED LIBRARIES
# ==============================================================================

import os
import random
import numpy as np
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta


# ==============================================================================
# INITIALIZE RANDOM GENERATORS
# ==============================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

fake = Faker("en_IN")
Faker.seed(SEED)

print("Libraries imported successfully.")
print(f"Random seed: {SEED}")

Libraries imported successfully.
Random seed: 42


In [17]:
# ==============================================================================
# PATIENT DATA CONFIGURATION
# ==============================================================================

TOTAL_PATIENTS = 1000

CSV_FILE_NAME = "dim_patient.csv"

# Output directory
OUTPUT_DIR = "03_Datasets/CSV_Files"

# Create output directory if it does not exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

CSV_FILE_PATH = os.path.join(
    OUTPUT_DIR,
    CSV_FILE_NAME
)

print("=" * 80)
print("PATIENT DATA CONFIGURATION")
print("=" * 80)
print(f"Total patients : {TOTAL_PATIENTS:,}")
print(f"Output file    : {CSV_FILE_PATH}")

PATIENT DATA CONFIGURATION
Total patients : 1,000
Output file    : 03_Datasets/CSV_Files\dim_patient.csv


## Patient Dimension Schema

The patient dimension stores relatively stable master information
about each patient.

The following fields are generated:

- patient_id
- patient_code
- first_name
- last_name
- gender
- date_of_birth
- age
- blood_group
- marital_status
- phone_number
- email
- address
- city
- state
- pincode
- emergency_contact_name
- emergency_contact_phone
- registration_date
- patient_status
- created_date
- last_updated


In [23]:
# ==============================================================================
# MASTER DATA VALUES
# ==============================================================================

GENDERS = [
    "Male",
    "Female"
]

BLOOD_GROUPS = [
    "A+",
    "A-",
    "B+",
    "B-",
    "AB+",
    "AB-",
    "O+",
    "O-"
]

MARITAL_STATUSES = [
    "Single",
    "Married",
    "Divorced",
    "Widowed"
]

PATIENT_STATUSES = [
    "Active",
    "Inactive"
]

# Major Indian states for realistic synthetic data
INDIAN_STATES = [
    "Maharashtra",
    "Madhya Pradesh",
    "Gujarat",
    "Karnataka",
    "Telangana",
    "Tamil Nadu",
    "Delhi",
    "Uttar Pradesh",
    "Rajasthan",
    "West Bengal"
]

print("Master data values configured successfully.")

Master data values configured successfully.


In [21]:
# ==============================================================================
# DATE CONFIGURATION
# ==============================================================================

TODAY = datetime(2026, 8, 4)

MIN_DATE_OF_BIRTH = TODAY - timedelta(days=365 * 90)
MAX_DATE_OF_BIRTH = TODAY - timedelta(days=365 * 1)

MIN_REGISTRATION_DATE = datetime(2020, 1, 1)

print("=" * 80)
print("DATE CONFIGURATION")
print("=" * 80)
print(f"Birth date range         : {MIN_DATE_OF_BIRTH.date()} to {MAX_DATE_OF_BIRTH.date()}")
print(f"Registration start date  : {MIN_REGISTRATION_DATE.date()}")
print(f"Reference date           : {TODAY.date()}")


DATE CONFIGURATION
Birth date range         : 1936-08-26 to 2025-08-04
Registration start date  : 2020-01-01
Reference date           : 2026-08-04


In [25]:
# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================

def calculate_age(date_of_birth, reference_date=TODAY):
    """
    Calculate patient age from date of birth.
    """

    age = (
        reference_date.year
        - date_of_birth.year
        - (
            (reference_date.month, reference_date.day)
            < (date_of_birth.month, date_of_birth.day)
        )
    )

    return age


def generate_phone_number():
    """
    Generate a realistic 10-digit Indian mobile number.
    """

    first_digit = random.choice(
        ["6", "7", "8", "9"]
    )

    remaining_digits = "".join(
        random.choices(
            "0123456789",
            k=9
        )
    )

    return first_digit + remaining_digits


def generate_patient_code(patient_id):
    """
    Generate unique patient business code.
    """

    return f"PAT{patient_id:06d}"


print("Helper functions created successfully.")

Helper functions created successfully.


In [26]:
# ==============================================================================
# GENERATE PATIENT RECORDS
# ==============================================================================

patient_records = []

for patient_id in range(1, TOTAL_PATIENTS + 1):

    # --------------------------------------------------------------------------
    # BASIC DEMOGRAPHICS
    # --------------------------------------------------------------------------

    gender = random.choice(GENDERS)

    first_name = fake.first_name_male() if gender == "Male" else fake.first_name_female()

    last_name = fake.last_name()

    date_of_birth = fake.date_between(
        start_date=MIN_DATE_OF_BIRTH.date(),
        end_date=MAX_DATE_OF_BIRTH.date()
    )

    date_of_birth = datetime.combine(
        date_of_birth,
        datetime.min.time()
    )

    age = calculate_age(date_of_birth)

    blood_group = random.choice(BLOOD_GROUPS)

    marital_status = random.choice(MARITAL_STATUSES)

    # --------------------------------------------------------------------------
    # CONTACT INFORMATION
    # --------------------------------------------------------------------------

    phone_number = generate_phone_number()

    email = (
        f"{first_name.lower()}."
        f"{last_name.lower()}."
        f"{patient_id}@example.com"
    )

    # --------------------------------------------------------------------------
    # ADDRESS INFORMATION
    # --------------------------------------------------------------------------

    address = fake.street_address()

    city = fake.city()

    state = random.choice(INDIAN_STATES)

    pincode = fake.postcode()

    # --------------------------------------------------------------------------
    # EMERGENCY CONTACT
    # --------------------------------------------------------------------------

    emergency_contact_name = fake.name()

    emergency_contact_phone = generate_phone_number()

    # --------------------------------------------------------------------------
    # REGISTRATION INFORMATION
    # --------------------------------------------------------------------------

    registration_date = fake.date_between(
        start_date=MIN_REGISTRATION_DATE.date(),
        end_date=TODAY.date()
    )

    registration_date = datetime.combine(
        registration_date,
        datetime.min.time()
    )

    patient_status = random.choices(
        PATIENT_STATUSES,
        weights=[95, 5],
        k=1
    )[0]

    # --------------------------------------------------------------------------
    # AUDIT INFORMATION
    # --------------------------------------------------------------------------

    created_date = registration_date

    last_updated = fake.date_time_between(
        start_date=created_date,
        end_date=TODAY
    )

    # --------------------------------------------------------------------------
    # CREATE RECORD
    # --------------------------------------------------------------------------

    patient_records.append(
        {
            "patient_id": patient_id,
            "patient_code": generate_patient_code(patient_id),
            "first_name": first_name,
            "last_name": last_name,
            "gender": gender,
            "date_of_birth": date_of_birth,
            "age": age,
            "blood_group": blood_group,
            "marital_status": marital_status,
            "phone_number": phone_number,
            "email": email,
            "address": address,
            "city": city,
            "state": state,
            "pincode": pincode,
            "emergency_contact_name": emergency_contact_name,
            "emergency_contact_phone": emergency_contact_phone,
            "registration_date": registration_date,
            "patient_status": patient_status,
            "created_date": created_date,
            "last_updated": last_updated
        }
    )


print(f"Generated {len(patient_records):,} patient records.")


Generated 1,000 patient records.


In [27]:
# ==============================================================================
# CREATE PATIENT DATAFRAME
# ==============================================================================

patient_df = pd.DataFrame(
    patient_records
)

# Ensure date columns have datetime data types
date_columns = [
    "date_of_birth",
    "registration_date",
    "created_date",
    "last_updated"
]

for column in date_columns:
    patient_df[column] = pd.to_datetime(
        patient_df[column]
    )


print("=" * 80)
print("PATIENT DATAFRAME CREATED")
print("=" * 80)

print(f"Rows    : {patient_df.shape[0]:,}")
print(f"Columns : {patient_df.shape[1]}")

patient_df.head()

PATIENT DATAFRAME CREATED
Rows    : 1,000
Columns : 21


,patient_id,patient_code,first_name,last_name,gender,date_of_birth,age,blood_group,marital_status,phone_number,...,address,city,state,pincode,emergency_contact_name,emergency_contact_phone,registration_date,patient_status,created_date,last_updated
0,1,PAT000001,Daksh,Bakshi,Male,1974-01-30,52,A+,Divorced,7276804025,...,"218, Sami Zila",Ludhiana,Maharashtra,338908,Oni Kannan,7774428714,2023-10-27,Active,2023-10-27,2026-04-29 09:25:01
1,2,PAT000002,Liam,Ahuja,Male,1958-05-19,68,AB-,Single,6333204190,...,H.No. 42\nNayar Ganj,Tiruchirappalli,Telangana,155940,Jeet Radhakrishnan,8570220212,2020-09-01,Active,2020-09-01,2025-05-13 03:07:59
2,3,PAT000003,Nandini,Pathak,Female,2015-03-25,11,B+,Divorced,8229661713,...,"10/341, Chaudhuri Ganj",Alwar,Rajasthan,525534,Onveer Walla,7687203229,2026-03-15,Active,2026-03-15,2026-05-01 12:54:05
3,4,PAT000004,Gautami,Shere,Female,1988-04-17,38,B-,Widowed,9861175745,...,83/503\nMutti Nagar,Agartala,Tamil Nadu,395376,Luke Lanka,7914781165,2021-03-09,Active,2021-03-09,2023-04-14 05:52:47
4,5,PAT000005,Lekha,Sangha,Female,2016-01-26,10,AB+,Single,6652614498,...,H.No. 32\nTak Circle,Nashik,Telangana,226916,Upadhriti Wadhwa,7516851319,2022-02-20,Active,2022-02-20,2022-03-27 16:05:13


In [28]:
# ==============================================================================
# DATA TYPE CHECK
# ==============================================================================

print("=" * 80)
print("DATA TYPES")
print("=" * 80)

print(
    patient_df.dtypes
)

DATA TYPES
patient_id                          int64
patient_code                       object
first_name                         object
last_name                          object
gender                             object
date_of_birth              datetime64[ns]
age                                 int64
blood_group                        object
marital_status                     object
phone_number                       object
email                              object
address                            object
city                               object
state                              object
pincode                            object
emergency_contact_name             object
emergency_contact_phone            object
registration_date          datetime64[ns]
patient_status                     object
created_date               datetime64[ns]
last_updated               datetime64[ns]
dtype: object


In [29]:
# ==============================================================================
# PATIENT DATA VALIDATION
# ==============================================================================

print("=" * 80)
print("PATIENT DATA VALIDATION")
print("=" * 80)


# ------------------------------------------------------------------------------
# 1. Row count
# ------------------------------------------------------------------------------

assert len(patient_df) == TOTAL_PATIENTS

print("✓ Row count validation passed.")


# ------------------------------------------------------------------------------
# 2. Patient ID uniqueness
# ------------------------------------------------------------------------------

assert patient_df["patient_id"].is_unique

print("✓ Patient ID uniqueness validation passed.")


# ------------------------------------------------------------------------------
# 3. Patient code uniqueness
# ------------------------------------------------------------------------------

assert patient_df["patient_code"].is_unique

print("✓ Patient code uniqueness validation passed.")


# ------------------------------------------------------------------------------
# 4. Required fields should not be null
# ------------------------------------------------------------------------------

required_columns = [
    "patient_id",
    "patient_code",
    "first_name",
    "last_name",
    "gender",
    "date_of_birth",
    "age",
    "blood_group",
    "phone_number",
    "registration_date"
]

for column in required_columns:

    assert patient_df[column].notna().all(), (
        f"Null values found in {column}"
    )

print("✓ Required field validation passed.")


# ------------------------------------------------------------------------------
# 5. Gender validation
# ------------------------------------------------------------------------------

assert patient_df["gender"].isin(
    GENDERS
).all()

print("✓ Gender validation passed.")


# ------------------------------------------------------------------------------
# 6. Blood group validation
# ------------------------------------------------------------------------------

assert patient_df["blood_group"].isin(
    BLOOD_GROUPS
).all()

print("✓ Blood group validation passed.")


# ------------------------------------------------------------------------------
# 7. Age validation
# ------------------------------------------------------------------------------

assert (
    (patient_df["age"] >= 1)
    &
    (patient_df["age"] <= 90)
).all()

print("✓ Age validation passed.")


# ------------------------------------------------------------------------------
# 8. Registration date validation
# ------------------------------------------------------------------------------

assert (
    patient_df["registration_date"]
    <= TODAY
).all()

print("✓ Registration date validation passed.")


# ------------------------------------------------------------------------------
# 9. Date of birth validation
# ------------------------------------------------------------------------------

assert (
    patient_df["date_of_birth"]
    <= TODAY
).all()

print("✓ Date of birth validation passed.")


# ------------------------------------------------------------------------------
# 10. Patient code format
# ------------------------------------------------------------------------------

assert patient_df["patient_code"].str.match(
    r"^PAT\d{6}$"
).all()

print("✓ Patient code format validation passed.")


print()
print("All patient data validations passed successfully. ✓")

PATIENT DATA VALIDATION
✓ Row count validation passed.
✓ Patient ID uniqueness validation passed.
✓ Patient code uniqueness validation passed.
✓ Required field validation passed.
✓ Gender validation passed.
✓ Blood group validation passed.
✓ Age validation passed.
✓ Registration date validation passed.
✓ Date of birth validation passed.
✓ Patient code format validation passed.

All patient data validations passed successfully. ✓


In [30]:
# ==============================================================================
# CHECK DUPLICATE RECORDS
# ==============================================================================

duplicate_count = patient_df.duplicated().sum()

print("=" * 80)
print("DUPLICATE CHECK")
print("=" * 80)

print(f"Duplicate records: {duplicate_count}")

assert duplicate_count == 0

print("✓ No duplicate records found.")

DUPLICATE CHECK
Duplicate records: 0
✓ No duplicate records found.


In [31]:
# ==============================================================================
# DATA QUALITY SUMMARY
# ==============================================================================

print("=" * 80)
print("DATA QUALITY SUMMARY")
print("=" * 80)

quality_summary = pd.DataFrame(
    {
        "Metric": [
            "Total Patients",
            "Total Columns",
            "Duplicate Rows",
            "Null Values",
            "Unique Patient IDs",
            "Unique Patient Codes"
        ],
        "Value": [
            len(patient_df),
            len(patient_df.columns),
            patient_df.duplicated().sum(),
            patient_df.isnull().sum().sum(),
            patient_df["patient_id"].nunique(),
            patient_df["patient_code"].nunique()
        ]
    }
)

quality_summary


DATA QUALITY SUMMARY


,Metric,Value
0,Total Patients,1000
1,Total Columns,21
2,Duplicate Rows,0
3,Null Values,0
4,Unique Patient IDs,1000
5,Unique Patient Codes,1000


In [32]:
# ==============================================================================
# PATIENT DEMOGRAPHIC SUMMARY
# ==============================================================================

print("=" * 80)
print("PATIENT DEMOGRAPHIC SUMMARY")
print("=" * 80)

print("\nGender distribution:")
print(
    patient_df["gender"]
    .value_counts()
)


print("\nBlood group distribution:")
print(
    patient_df["blood_group"]
    .value_counts()
)


print("\nMarital status distribution:")
print(
    patient_df["marital_status"]
    .value_counts()
)


print("\nPatient status distribution:")
print(
    patient_df["patient_status"]
    .value_counts()
)


PATIENT DEMOGRAPHIC SUMMARY

Gender distribution:
gender
Male      525
Female    475
Name: count, dtype: int64

Blood group distribution:
blood_group
O+     139
A-     135
O-     129
B+     125
AB-    124
B-     120
A+     114
AB+    114
Name: count, dtype: int64

Marital status distribution:
marital_status
Single      275
Divorced    255
Widowed     240
Married     230
Name: count, dtype: int64

Patient status distribution:
patient_status
Active      954
Inactive     46
Name: count, dtype: int64


In [33]:
# ==============================================================================
# PATIENT AGE SUMMARY
# ==============================================================================

print("=" * 80)
print("PATIENT AGE SUMMARY")
print("=" * 80)

print(
    patient_df["age"].describe()
)

PATIENT AGE SUMMARY
count    1000.000000
mean       45.036000
std        25.722137
min         1.000000
25%        22.000000
50%        45.000000
75%        68.000000
max        89.000000
Name: age, dtype: float64


SyntaxError: invalid syntax (2974402961.py, line 1)